# AC-RSVD numerical experiments

This notebook is the Python driver for the numerical section. It calls the C++ algorithms through the PyTorch binding and keeps the experimental decisions in one place.

The current binding supports dense pilot runs. The formal `n = 2**17` comparison still needs the structured Hadamard operator, C++ phase timers, certificate traces, ablation entry points, and theorem-bound evaluators. Dense pilot results are never used as paper results.

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.stats import beta

ROOT = Path.cwd()
if ROOT.name == "experiments":
    ROOT = ROOT.parent

RESULTS = ROOT / "experiments" / "results"
FIGURES = ROOT / "experiments" / "figures"
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
torch.ops.load_library(str(ROOT / "build" / "ac_rsvd_torch.so"))

sns.set_theme(style="whitegrid", context="paper")
COLORS = sns.color_palette("mako", n_colors=6)
SAVE_DPI = 600
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": SAVE_DPI,
    "font.family": "serif",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "lines.linewidth": 1.6,
    "lines.markersize": 4,
    "grid.alpha": 0.25,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
})

## Fixed experimental plan

The run counts and dimensions come from the v6 numerical plan. The spectrum formulas below fill the two choices that v6 left open. Before the paper run, these formulas are frozen together with the commit hash.

For theory bounds, the candidate parameter grids are fixed before any random path is examined. The bound evaluator will choose the smallest published finite bound using only the known spectrum and tolerance.

In [ ]:
CONFIG = {
    "dtype": "float64",
    "block_size": 32,
    "matrix_seed": 20260817,
    "main_delta": 0.01,
    "reliability_n": 256,
    "reliability_target_rank": 64,
    "main_n": 2**17,
    "main_target_ranks": [64, 192],
    "main_seeds": list(range(10000, 10005)),
    "ablation_seeds": [10000, 10001, 10002],
    "eta_r": 0.01,
    "eta_c": 0.01,
    "rho_grid": [0.35, 0.45, 0.55, 0.65, 0.75, 0.85],
    "alpha_grid": [0.30, 0.40, 0.50, 0.60, 0.70],
}

FORMAL_RELIABILITY_GROUPS = pd.DataFrame([
    {"spectrum": "smooth", "delta": 0.1, "seed_start": 0, "runs": 500},
    {"spectrum": "clustered", "delta": 0.1, "seed_start": 500, "runs": 500},
    {"spectrum": "smooth", "delta": 0.01, "seed_start": 1000, "runs": 2000},
    {"spectrum": "clustered", "delta": 0.01, "seed_start": 3000, "runs": 2000},
])

FORMAL_RELIABILITY_GROUPS

## Known spectra and dense pilot matrices

The smooth spectrum is `sigma_i proportional to i**(-1)`. The clustered spectrum replaces 32 values around the target rank by a flat shoulder. Every spectrum is scaled to unit Frobenius norm.

The tolerance is the midpoint between the squared tails at ranks `r - 1` and `r`, so the exact optimal tolerance rank is `r`. The dense Hadamard matrices below are only for the small pilot. The formal operator will apply the same signs, permutations, and transforms without materializing the matrix.

In [ ]:
def make_spectrum(n, target_rank, kind):
    index = torch.arange(1, n + 1, dtype=torch.float64)
    values = 1.0 / index
    if kind == "clustered":
        left = target_rank - 16
        right = target_rank + 16
        values[left:right] = 1.0 / target_rank
    return values / torch.linalg.vector_norm(values)


def squared_tails(singular_values):
    tail = torch.empty(singular_values.numel() + 1, dtype=torch.float64)
    tail[-1] = 0.0
    tail[:-1] = torch.flip(
        torch.cumsum(torch.flip(singular_values.square(), dims=[0]), dim=0),
        dims=[0],
    )
    return tail


def tolerance_for_rank(singular_values, target_rank):
    tail = squared_tails(singular_values)
    tolerance_squared = tail[target_rank] + 0.5 * singular_values[target_rank - 1].square()
    return torch.sqrt(tolerance_squared).item()


def optimal_rank(singular_values, tolerance):
    tail = squared_tails(singular_values)
    return int(torch.nonzero(tail <= tolerance * tolerance)[0, 0])


def dense_hadamard(n):
    matrix = torch.ones((1, 1), dtype=torch.float64)
    while matrix.shape[0] < n:
        top = torch.cat((matrix, matrix), dim=1)
        bottom = torch.cat((matrix, -matrix), dim=1)
        matrix = torch.cat((top, bottom), dim=0) / np.sqrt(2.0)
    return matrix


def make_dense_matrix(singular_values, seed):
    n = singular_values.numel()
    hadamard = dense_hadamard(n)
    generator = torch.Generator().manual_seed(seed)

    left_sign = 2 * torch.randint(0, 2, (n,), generator=generator) - 1
    right_sign = 2 * torch.randint(0, 2, (n,), generator=generator) - 1
    left_order = torch.randperm(n, generator=generator)
    right_order = torch.randperm(n, generator=generator)

    left = left_sign[:, None] * hadamard[:, left_order]
    right = right_sign[:, None] * hadamard[:, right_order]
    return ((left * singular_values) @ right.T).contiguous()

In [ ]:
spectrum_checks = []
for kind in ["smooth", "clustered"]:
    singular_values = make_spectrum(256, 64, kind)
    tolerance = tolerance_for_rank(singular_values, 64)
    spectrum_checks.append({
        "spectrum": kind,
        "frobenius_norm": torch.linalg.vector_norm(singular_values).item(),
        "tolerance": tolerance,
        "optimal_rank": optimal_rank(singular_values, tolerance),
    })

pd.DataFrame(spectrum_checks)

## PyTorch binding

Every call below runs one complete C++ algorithm. `binding_wall_seconds` includes Python/C++ copies and is only a pilot diagnostic. Paper timings will come from C++ phase timers.

In [ ]:
STOP_NAMES = {
    0: "certificate",
    1: "tolerance_met",
    2: "full_output_space",
    3: "full_rank",
    4: "final_input_direction",
    5: "zero_residual",
}


def run_method(method, matrix, singular_values, tolerance, delta, block_size, seed):
    start = perf_counter()
    if method == "AC-RSVD":
        result = torch.ops.ac_rsvd.run_ac_rsvd(
            matrix, tolerance, delta, block_size, seed, 0
        )
    elif method == "randQB_MF_Fro":
        result = torch.ops.ac_rsvd.run_randqb_mf_fro(
            matrix, tolerance, block_size, seed, 0
        )
    else:
        frobenius_norm = torch.linalg.vector_norm(singular_values).item()
        result = torch.ops.ac_rsvd.run_randqb_ei(
            matrix, tolerance, frobenius_norm, block_size, seed, 0
        )
    binding_wall_seconds = perf_counter() - start

    u, values, v, counters, diagnostics, stop_reason = result
    approximation = (u * values) @ v.T
    error = torch.linalg.matrix_norm(matrix - approximation).item()
    if values.numel() == 0:
        orthogonality_error = 0.0
    else:
        gram = u.T @ u
        identity = torch.eye(values.numel(), dtype=torch.float64)
        orthogonality_error = torch.linalg.matrix_norm(gram - identity).item()

    counter_values = counters.tolist()
    diagnostic_values = diagnostics.tolist()
    return {
        "algorithm": method,
        "seed": seed,
        "rank": values.numel(),
        "error_fro": error,
        "error_over_tau": error / tolerance,
        "directions": counter_values[0],
        "a_columns": counter_values[1],
        "at_columns": counter_values[2],
        "a_block_calls": counter_values[3],
        "at_block_calls": counter_values[4],
        "residual_estimate_sq": diagnostic_values[0],
        "residual_bound_sq": diagnostic_values[1],
        "truncation_budget_sq": diagnostic_values[2],
        "output_orthogonality_error": orthogonality_error,
        "stop_reason": STOP_NAMES[int(stop_reason)],
        "binding_wall_seconds": binding_wall_seconds,
    }

## Smoke test

This checks the three binding paths on the same matrix. It is not part of a paper table.

In [ ]:
smoke_spectrum = make_spectrum(64, 16, "smooth")
smoke_tolerance = tolerance_for_rank(smoke_spectrum, 16)
smoke_matrix = make_dense_matrix(smoke_spectrum, CONFIG["matrix_seed"])
smoke_rows = [
    run_method(method, smoke_matrix, smoke_spectrum, smoke_tolerance, 0.1, 8, 7)
    for method in ["AC-RSVD", "randQB_MF_Fro", "randQB_EI"]
]
smoke_results = pd.DataFrame(smoke_rows)
smoke_results[[
    "algorithm", "rank", "error_over_tau", "directions",
    "a_columns", "at_columns", "at_block_calls", "stop_reason",
]]

## Experiment 1: final-error failure probability

The formal grid has 5,000 AC-RSVD paths on two fixed `256 x 256` matrices. A failure means the direct compact-SVD Frobenius error is larger than the requested tolerance. The one-sided interval is the exact 95% Clopper-Pearson upper bound.

The small pilot below checks the complete CSV and Figure 1 path. Formal execution stays off by default.

In [ ]:
def run_reliability(groups):
    matrices = {}
    spectra = {}
    tolerances = {}
    for offset, kind in enumerate(["smooth", "clustered"]):
        singular_values = make_spectrum(
            CONFIG["reliability_n"], CONFIG["reliability_target_rank"], kind
        )
        spectra[kind] = singular_values
        tolerances[kind] = tolerance_for_rank(
            singular_values, CONFIG["reliability_target_rank"]
        )
        matrices[kind] = make_dense_matrix(
            singular_values, CONFIG["matrix_seed"] + offset
        )

    rows = []
    for group in groups.itertuples(index=False):
        for repetition in range(group.runs):
            seed = group.seed_start + repetition
            tolerance = tolerances[group.spectrum]
            row = run_method(
                "AC-RSVD",
                matrices[group.spectrum],
                spectra[group.spectrum],
                tolerance,
                group.delta,
                CONFIG["block_size"],
                seed,
            )
            row.update({
                "experiment": "reliability",
                "spectrum": group.spectrum,
                "n": CONFIG["reliability_n"],
                "target_rank": CONFIG["reliability_target_rank"],
                "optimal_rank": CONFIG["reliability_target_rank"],
                "tolerance": tolerance,
                "delta": group.delta,
                "repetition": repetition,
                "failed": row["error_fro"] > tolerance,
            })
            rows.append(row)
    return pd.DataFrame(rows)


def clopper_pearson_upper(failures, runs):
    if failures == runs:
        return 1.0
    return beta.ppf(0.95, failures + 1, runs - failures)


def summarize_reliability(runs):
    summary = (
        runs.groupby(["spectrum", "delta"], as_index=False)
        .agg(failures=("failed", "sum"), runs=("failed", "size"))
    )
    summary["failure_rate"] = summary["failures"] / summary["runs"]
    summary["upper_95"] = [
        clopper_pearson_upper(int(row.failures), int(row.runs))
        for row in summary.itertuples(index=False)
    ]
    return summary


def plot_failure_probability(summary, path):
    figure, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), sharey=True)
    y_limit = min(1.0, max(0.12, 1.12 * summary["upper_95"].max()))
    for axis, kind in zip(axes, ["smooth", "clustered"]):
        data = summary[summary["spectrum"] == kind].sort_values("delta")
        x = data["delta"].to_numpy()
        y = data["failure_rate"].to_numpy()
        upper = data["upper_95"].to_numpy()
        axis.errorbar(
            x, y, yerr=np.vstack((np.zeros_like(y), upper - y)),
            color=COLORS[4], marker="o", capsize=3, label="Observed",
        )
        axis.plot(x, x, color=COLORS[1], linestyle="--", label="Requested delta")
        for row in data.itertuples(index=False):
            axis.annotate(
                f"{int(row.failures)}/{int(row.runs)}",
                (row.delta, row.failure_rate), xytext=(0, 6),
                textcoords="offset points", ha="center", fontsize=8,
            )
        axis.set_xscale("log")
        axis.set_xticks([0.01, 0.1], labels=["0.01", "0.1"])
        axis.set_ylim(0.0, y_limit)
        axis.set_title(kind.capitalize())
        axis.set_xlabel("Requested failure probability")
    axes[0].set_ylabel("Empirical failure probability")
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(
        handles, labels, loc="upper center", ncol=2, frameon=False,
        bbox_to_anchor=(0.5, 1.0),
    )
    sns.despine(figure)
    figure.tight_layout(rect=(0.0, 0.0, 1.0, 0.88))
    figure.savefig(path, dpi=SAVE_DPI, facecolor="white")
    return figure

In [ ]:
PILOT_RELIABILITY_GROUPS = pd.DataFrame([
    {"spectrum": "smooth", "delta": 0.1, "seed_start": 0, "runs": 3},
    {"spectrum": "clustered", "delta": 0.1, "seed_start": 100, "runs": 3},
    {"spectrum": "smooth", "delta": 0.01, "seed_start": 200, "runs": 5},
    {"spectrum": "clustered", "delta": 0.01, "seed_start": 300, "runs": 5},
])

pilot_reliability_runs = run_reliability(PILOT_RELIABILITY_GROUPS)
pilot_reliability_summary = summarize_reliability(pilot_reliability_runs)
pilot_reliability_runs.to_csv(RESULTS / "pilot_reliability_runs.csv", index=False)
pilot_reliability_summary.to_csv(RESULTS / "pilot_reliability_summary.csv", index=False)
plot_failure_probability(
    pilot_reliability_summary, FIGURES / "pilot_figure_1_failure_probability.png"
)
pilot_reliability_summary

In [ ]:
RUN_FORMAL_RELIABILITY = False

if RUN_FORMAL_RELIABILITY:
    reliability_runs = run_reliability(FORMAL_RELIABILITY_GROUPS)
    reliability_summary = summarize_reliability(reliability_runs)
    reliability_runs.to_csv(RESULTS / "reliability_runs.csv", index=False)
    reliability_summary.to_csv(RESULTS / "reliability_summary.csv", index=False)
    plot_failure_probability(
        reliability_summary, FIGURES / "figure_1_failure_probability.png"
    )
else:
    print("Formal reliability run is disabled.")

## Experiment 2: three-method comparison

The formal grid has 60 paired runs: two spectra, two optimal ranks, five seeds, and three algorithms. `randQB_EI` receives the exact norm `1.0`.

This manifest is fixed now. Execution waits for the structured C++ operator and internal timers. A dense `2**17` tensor would require 128 GiB before the binding makes its extra copy.

In [ ]:
comparison_manifest = pd.DataFrame([
    {
        "experiment": "comparison",
        "spectrum": spectrum,
        "target_rank": target_rank,
        "algorithm": algorithm,
        "seed": seed,
        "n": CONFIG["main_n"],
        "delta": CONFIG["main_delta"] if algorithm == "AC-RSVD" else np.nan,
        "block_size": CONFIG["block_size"],
        "threads": 10,
    }
    for spectrum in ["smooth", "clustered"]
    for target_rank in CONFIG["main_target_ranks"]
    for algorithm in ["AC-RSVD", "randQB_MF_Fro", "randQB_EI"]
    for seed in CONFIG["main_seeds"]
])

comparison_manifest.groupby(["spectrum", "target_rank", "algorithm"]).size()

### Table 1 organization

`comparison_runs.csv` keeps one row per run. Table 1 has one row per `(spectrum, target_rank, algorithm)` and reports the median of the five paired seeds. Individual points remain in the raw CSV.

The formal row includes direct error, output rank, operator columns and calls, unused forward columns, and C++ total/phase times. `binding_wall_seconds` is excluded.

In [ ]:
TABLE_1_COLUMNS = [
    "spectrum", "target_rank", "algorithm", "tolerance", "runs",
    "frobenius_norm_source", "error_over_tau", "rank_over_optimal",
    "a_columns", "at_columns", "a_block_calls", "at_block_calls",
    "unused_a_columns", "time_total_s", "time_a_s", "time_at_s",
    "time_orth_s", "time_certificate_s", "time_svd_s",
]

TABLE_1_COLUMNS

## Theorem 2 and Corollary 2.1

Table 2 reuses the 20 AC-RSVD comparison paths. It reports actual directions, the full-spectrum Theorem 2 bound, the rank-only Corollary 2.1 bound, and the deterministic cap.

The candidate grids below are deterministic. For each known spectrum and tolerance, the evaluator minimizes the published finite bound before looking at the random run. The evaluator is still missing from the C++ core.

In [ ]:
analysis_parameter_grid = pd.DataFrame([
    {"alpha": alpha, "rho": rho, "eta_r": CONFIG["eta_r"], "eta_c": CONFIG["eta_c"]}
    for alpha in CONFIG["alpha_grid"]
    for rho in CONFIG["rho_grid"]
    if alpha * alpha < rho
])

TABLE_2_COLUMNS = [
    "spectrum", "target_rank", "seed", "tolerance", "delta",
    "directions_actual", "theorem2_bound", "theorem2_over_actual",
    "corollary21_bound", "corollary21_over_actual",
    "deterministic_cap", "deterministic_over_actual",
    "theorem2_k", "theorem2_p", "corollary_alpha",
    "corollary_p", "rho", "eta_r", "eta_c",
]

analysis_parameter_grid.head()

## Experiment 3: paired ablations

The ablation grid has two spectra, target rank 192, and three paired seeds. The sequential orthogonalization variant adds six runs. Continuous inversion and grid rounding reuse the same six certificate traces. Terminal adjoint calls stay in Table 1.

Figure 2 uses six panels: three block-orthogonalization metrics and three inversion metrics. Every line connects the same matrix and random seed.

In [ ]:
ablation_manifest = pd.DataFrame([
    {
        "spectrum": spectrum,
        "target_rank": 192,
        "seed": seed,
        "pair_id": f"{spectrum}-{seed}",
        "n": CONFIG["main_n"],
        "delta": CONFIG["main_delta"],
        "block_size": CONFIG["block_size"],
    }
    for spectrum in ["smooth", "clustered"]
    for seed in CONFIG["ablation_seeds"]
])

BLOCK_ABLATION_COLUMNS = [
    "pair_id", "spectrum", "seed", "variant",
    "time_orth_s", "time_total_s", "orthogonality_error",
    "rank", "error_fro",
]
INVERSE_ABLATION_COLUMNS = [
    "pair_id", "spectrum", "seed", "variant",
    "bound_over_tau_sq", "budget_over_tau_sq", "rank_over_optimal",
]

ablation_manifest

In [ ]:
def plot_paired_ablations(block_runs, inverse_runs, path):
    figure, axes = plt.subplots(2, 3, figsize=(7.2, 5.2))
    block_metrics = [
        ("time_orth_s", "Orthogonalization time (s)"),
        ("time_total_s", "Total time (s)"),
        ("orthogonality_error", "Orthogonality error"),
    ]
    inverse_metrics = [
        ("bound_over_tau_sq", "Residual bound / tau^2"),
        ("budget_over_tau_sq", "Truncation budget / tau^2"),
        ("rank_over_optimal", "Output rank / optimal rank"),
    ]

    for axis, (metric, title) in zip(axes[0], block_metrics):
        for _, pair in block_runs.groupby("pair_id"):
            values = pair.set_index("variant").loc[["sequential", "blocked"], metric]
            axis.plot([0, 1], values, color=COLORS[3], alpha=0.65, marker="o")
        axis.set_xticks([0, 1], labels=["Sequential", "Blocked"])
        axis.set_title(title)

    for axis, (metric, title) in zip(axes[1], inverse_metrics):
        for _, pair in inverse_runs.groupby("pair_id"):
            values = pair.set_index("variant").loc[["grid", "continuous"], metric]
            axis.plot([0, 1], values, color=COLORS[5], alpha=0.65, marker="o")
        axis.set_xticks([0, 1], labels=["Grid", "Continuous"])
        axis.set_title(title)

    sns.despine(figure)
    figure.tight_layout()
    figure.savefig(path, dpi=SAVE_DPI, facecolor="white")
    return figure

## Result files

Raw data stay tidy: one row per algorithm path or one row per paired variant. Summary tables are derived from raw CSV files.

```text
experiments/results/
  reliability_runs.csv
  reliability_summary.csv
  comparison_runs.csv
  block_ablation_runs.csv
  inverse_replay.csv
  theorem_bounds.csv
  table_1.csv
  table_2.csv

experiments/figures/
  figure_1_failure_probability.png      # 1 x 2, 600 DPI
  figure_2_paired_ablations.png         # 2 x 3, 600 DPI
```

Generated CSV and PNG files remain outside Git.

## Implementation readiness

| Part | Ready now | Required next |
|---|---:|---|
| Dense smoke test | yes | none |
| 256 x 256 reliability | yes | eight-process runner for final speed |
| Figure 1 and Clopper-Pearson summary | yes | formal 5,000-run CSV |
| n = 2**17 comparison | no | structured C++ operator and binding |
| Table 1 timing columns | no | C++ total and phase timers |
| Table 2 | no | Theorem 2 and Corollary 2.1 evaluators |
| Sequential/block ablation | no | sequential AC-RSVD entry point |
| Grid/continuous inversion ablation | no | certificate trace and replay output |
| Strict floating-point certificate | no | Appendix F enclosure path |